## 9.5 文書検索モデルとChatGPTを組み合わせる

### 9.5.1 検索モデルの準備

In [1]:
!pip install 'datasets<4.0.0' openai==0.27 tiktoken 'transformers[ja]<4.41.0'  faiss-cpu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 9.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 105.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 45.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.1/70.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 694.9/694.9 kB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 MB 29.8 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset
from transformers import pipeline

dataset_name = "llm-book/aio-passages-bpr-bert-base-japanese-v3"
passage_dataset = load_dataset(dataset_name, split="train")

encoder_model_name = "llm-book/bert-base-japanese-v3-bpr-question-aio"
encoder_pipeline = pipeline(
    "feature-extraction", model=encoder_model_name, device="cuda:0"
)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-511e49a90d2afb(…):   0%|          | 0.00/343M [00:00<?, ?B/s]

data/train-00001-of-00007-6c92cf6fc0e424(…):   0%|          | 0.00/328M [00:00<?, ?B/s]

data/train-00002-of-00007-0dbf15822023e3(…):   0%|          | 0.00/307M [00:00<?, ?B/s]

data/train-00003-of-00007-fd06e9e09d6bd8(…):   0%|          | 0.00/304M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/445M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/529 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [3]:
print(passage_dataset)

Dataset({
    features: ['id', 'pageid', 'revid', 'text', 'section', 'title', 'embeddings'],
    num_rows: 4288198
})


In [4]:
print(passage_dataset[0]["embeddings"])

[133, 162, 145, 21, 151, 215, 254, 119, 214, 80, 4, 189, 177, 53, 100, 115, 68, 177, 70, 103, 41, 90, 127, 227, 113, 27, 71, 148, 92, 162, 176, 133, 105, 99, 16, 16, 52, 70, 132, 46, 2, 32, 211, 149, 29, 103, 53, 233, 29, 199, 124, 65, 178, 90, 60, 32, 201, 114, 214, 132, 60, 254, 216, 249, 184, 57, 119, 181, 23, 253, 121, 83, 63, 115, 141, 73, 90, 0, 239, 225, 194, 248, 16, 108, 66, 215, 124, 248, 0, 26, 214, 78, 52, 118, 115, 194]


In [5]:
print(dir(passage_dataset))

['_TF_DATASET_REFS', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__enter__', '__eq__', '__exit__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getitems__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setstate__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_build_local_temp_path', '_check_index_is_initialized', '_data', '_estimate_nbytes', '_fingerprint', '_format_columns', '_format_kwargs', '_format_type', '_generate_tables_from_cache_file', '_generate_tables_from_shards', '_get_cache_file_path', '_get_output_signature', '_getitem', '_indexes', '_indices', '_info', '_map_single', '_new_dataset_with_indices', '_output_all_columns', '_push_parquet_shards_to_hub', '_save_to_disk_single', '_select_contiguous', '_select_with_indices_mapping', '_split', 'add_colum

#### `passage_dataset["embeddings"]`を`Faiss`ライブラリのインデックスに格納する

In [6]:
import faiss
import numpy as np
from tqdm import tqdm

# インデックスの初期化
embed_size = encoder_pipeline.model.config.hidden_size
faiss_index = faiss.IndexBinaryIDMap2(
    faiss.IndexBinaryFlat(embed_size)
)

with tqdm(total=len(passage_dataset)) as pbar:
    i = 0
    for batch in passage_dataset.iter(batch_size=512):
        bs = len(batch["embeddings"])

        # 埋め込みをインデックスに適したdtypeへと変換
        batch_embeddings = np.array(
            batch["embeddings"], dtype=np.uint8
        )
        batch_indices = np.arange(i, i + bs, dtype=np.int64)

        # 埋め込みをインデックスに格納
        faiss_index.add_with_ids(batch_embeddings, batch_indices)

        pbar.update(n=bs)
        i += bs

100%|██████████| 4288198/4288198 [01:33<00:00, 45928.70it/s]


In [7]:
import torch

def embed_questions(questions: list[str]) -> np.ndarray:
    """質問文を実数ベクトルに変換"""
    output_tensors = encoder_pipeline(questions, return_tensors="pt")
    embeddings = np.stack([
        t.squeeze(0)[0].numpy().astype(np.float32)
        for t in output_tensors
    ])
    return embeddings

def binarize_embeddings(embeddings: np.ndarray) -> np.ndarray:
    """実数ベクトルをバイナリベクトルに変換"""
    # 0未満の値を0に、0以上の値を1に変換
    binary_embeddings = np.where(embeddings < 0, 0, 1)
    # バイト単位でまとめてuint8に変換
    packed_binary_embeddings = np.packbits(binary_embeddings, axis=1)
    return packed_binary_embeddings

In [8]:
q_embed = embed_questions(["日本で一番高い山は何？"])
binary_q_embed = binarize_embeddings(q_embed)
print(binary_q_embed)

[[141 210 243   2 153 138 188 209 212 205 166 253 146  26  12 135 215 208
    0 233 223  17  43   5 198 163   1 116 111  63 138 212 106 118 126  84
   31 166  23 194 190 164 201 165 139  83 132  46  86  11  97  24  81 227
   12  46 128 115  17  49  22 224 101  16 252 100 229  33  92  77 216 119
  176  62 252  97  16 110 203  36 195 208  32 170 138 159 255  83  83 122
   64 166 100 246  83  44]]


In [9]:
scores, passage_ids = faiss_index.search(binary_q_embed, k=5)
print(f"scores: {scores} \n passageId: {passage_ids}")

scores: [[168 169 170 173 174]] 
 passageId: [[2222502 2783191 2879647 3581406 1149955]]


In [10]:
print(passage_dataset[passage_ids[0]]["text"])

['日本の領土に占める山間部の割合はおよそ7割程度である。2020年現在、日本で一番高い山は富士山で、逆に一番低い山は日和山である。日本は新期造山帯に位置し、多くの火山が見られる。日本では昔から人々が山と共生する文化が培われてきた。それらの山は里山とよばれ、農村などでは山を共有地として村全体で管理し、薪をとったり土や山菜などを利用する目的で利用され、手入れをされてきた。しかし、明治時代になるとそれらの山の中には国有地にされてしまったものも多く存在する。また、日本の山の中には霊峰とよばれ、民間信仰の場にされた山も多くある。山の中には昔から金山や銀山として、近代では銅や石炭などを入手するため多くの鉱山が作られ、鉱毒や粉塵などが問題になった。特に近代において、山林の多くで木材を得るための大規模な伐採などが行われた。', '3000 m峰は、独立峰の富士山と御嶽山及び飛騨山脈(北アルプス)と赤石山脈(南アルプス)の山域に限られている。24番目に高い山は剱岳 (2999 m) で、3000 mにわずか1 m届いていない。標高3000 mを越える一帯は森林限界のハイマツ帯で、高山植物の群生地となっている。また多くの3000 m峰のハイマツ帯は、ライチョウ(雷鳥)の生息地となっている。', '三角点は一般的に眺望の利く場所に設置され一等三角点は970点余りに上るが、その中で風格のある山容、優れた眺望、高い知名度、さらに概ね標高1,000m以上で登りがいのある山が選定された。一等三角点の最高峰は、標高3,121 mの南アルプスの赤石岳である。三角点がその山の最高峰とは限らず、山頂に三角点より高い場所があり、その地点すなわち標高点や測定点を以て山の高さとしている例も少なくない(例:早池峰、御嶽山、金剛山)。また連山を形成している場合、最高峰でないピークに一等三角点が設置されている例もある(例:赤城山、穂高岳、阿蘇山)。特殊な例としては雲仙普賢岳のように、噴火により一等三角点が最高峰でなくなった事例もある。利尻岳中腹の長官山のように全く山頂とは離れた地点に一等三角点が設置されている例もある(山頂は二等三角点「利尻絶頂」)。', 'この項では世界最高峰と考えられていた山(せかいさいこうほうとかんがえられていたやま)について述べる。19世紀初頭までアンデスが世界で最も高い山脈だと考えられて

In [11]:
passage_id = 0
print(np.unpackbits(faiss_index.reconstruct(passage_id)))

[1 0 0 0 0 1 0 1 1 0 1 0 0 0 1 0 1 0 0 1 0 0 0 1 0 0 0 1 0 1 0 1 1 0 0 1 0
 1 1 1 1 1 0 1 0 1 1 1 1 1 1 1 1 1 1 0 0 1 1 1 0 1 1 1 1 1 0 1 0 1 1 0 0 1
 0 1 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 1 1 1 0 1 1 0 1 1 0 0 0 1 0 0 1 1 0 1 0
 1 0 1 1 0 0 1 0 0 0 1 1 1 0 0 1 1 0 1 0 0 0 1 0 0 1 0 1 1 0 0 0 1 0 1 0 0
 0 1 1 0 0 1 1 0 0 1 1 1 0 0 1 0 1 0 0 1 0 1 0 1 1 0 1 0 0 1 1 1 1 1 1 1 1
 1 1 0 0 0 1 1 0 1 1 1 0 0 0 1 0 0 0 1 1 0 1 1 0 1 0 0 0 1 1 1 1 0 0 1 0 1
 0 0 0 1 0 1 1 1 0 0 1 0 1 0 0 0 1 0 1 0 1 1 0 0 0 0 1 0 0 0 0 1 0 1 0 1 1
 0 1 0 0 1 0 1 1 0 0 0 1 1 0 0 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 1 0 1 0 0
 0 1 0 0 0 1 1 0 1 0 0 0 0 1 0 0 0 0 1 0 1 1 1 0 0 0 0 0 0 0 1 0 0 0 1 0 0
 0 0 0 1 1 0 1 0 0 1 1 1 0 0 1 0 1 0 1 0 0 0 1 1 1 0 1 0 1 1 0 0 1 1 1 0 0
 1 1 0 1 0 1 1 1 1 0 1 0 0 1 0 0 0 1 1 1 0 1 1 1 0 0 0 1 1 1 0 1 1 1 1 1 0
 0 0 1 0 0 0 0 0 1 1 0 1 1 0 0 1 0 0 1 0 1 1 0 1 0 0 0 1 1 1 1 0 0 0 0 1 0
 0 0 0 0 1 1 0 0 1 0 0 1 0 1 1 1 0 0 1 0 1 1 0 1 0 1 1 0 1 0 0 0 0 1 0 0 0
 0 1 1 1 1 0 0 1 1 1 1 1 

In [12]:
import numpy as np
import torch

def retrieve_passages(
    questions: list[str],
    binary_k: int=2048,  # リランキングするための候補の取得数
    top_k: int=3  # リランキング後に出力する最終的なパッセージ数
) -> list[list[str]]:
    """質問をバイナリベクトルに変換する"""
    q_embed = embed_questions(questions)
    q_bin_embed = binarize_embeddings(q_embed)

    # バイナリベクトルを使用して検索する
    binary_k_scores, binary_k_p_ids = faiss_index.search(
        q_bin_embed, binary_k
    )

    batch_size = len(questions)
    embed_dim = q_bin_embed.shape[-1] * 8
    # インデックスからパッセージのベクトルを復元
    p_uint8_embed = [
        faiss_index.reconstruct(int(p_id))
        for p_id in binary_k_p_ids.flatten()
    ]
    # uint8からboolに変換
    p_bin_embed = np.vstack([np.unpackbits(e) for e in p_uint8_embed])
    # 型とshapeを変換
    p_bin_embed = p_bin_embed.astype(np.float32).reshape(
        batch_size, binary_k, embed_dim
    )
    p_bin_embed = p_bin_embed * 2 - 1

    # 質問の実数ベクトルとパッセージのバイナリベクトルで再度スコアを計算する
    re_scores = np.einsum("ijk,ik->ij", p_bin_embed, q_embed)
    top_k_indices = np.argsort(-re_scores, axis=-1)[:, :top_k]
    top_k_p_ids = np.take_along_axis(
        binary_k_p_ids, top_k_indices, axis=-1
    )

    # top_kのテキストを整形して出力する
    retrieved_texts: list[list[str]] = []
    for p_ids in top_k_p_ids:
        formatted_texts = [
            f"タイトル: {passage_dataset[i]['title']}\n"
            f"本文: {passage_dataset[i]['text']}"
            for i in p_ids.tolist()
        ]
        retrieved_texts.append(formatted_texts)

    return retrieved_texts

In [13]:
passages = retrieve_passages(["神奈川県の県庁所在地は？"], top_k=3)[0]
for passage in passages:
    print(passage)

タイトル: 神奈川県
本文: 藤沢市は人口が30万人を超えているが(約43万人)、中核市に指定されていない。県東部の横浜市、川崎市は、都市化、工業化が進んでおり、東京湾に面した京浜工業地帯の一角を形成する。県西部は緑豊かな丹沢山地から足柄山地、箱根山が連なり、酒匂川が流れる足柄平野には小田原城の城下町・小田原市が開ける。県中央部は相模原市、厚木市、海老名市などの平野部で都市化・工業化が進んでおり、相模川が流れ平塚市から相模湾に注ぐ。県南東部は、海沿いに茅ヶ崎市、藤沢市が開け、鎌倉幕府が置かれた鎌倉市から、明治以来の軍港都市・横須賀市がある三浦半島にかけて、三浦丘陵が連なる。新都心の横浜みなとみらい21、横浜赤レンガ倉庫、横浜中華街といった都市部の観光地だけでなく、箱根、熱海、茅ヶ崎、江の島といった山や海のリゾート地にも恵まれている事から日本全国から観光客が訪れる県になっている。
タイトル: 神奈川県
本文: 神奈川県(かながわけん、(英: Kanagawa Prefecture)は、日本の関東地方に位置する県。県庁所在地は横浜市。
タイトル: 神奈川県私立中学高等学校協会
本文: 横浜市(神奈川区・中区・西区・保土ケ谷区 ・南区)横浜市(港北区・鶴見区・都筑区)横浜市(泉区・磯子区・金沢区・港南区・栄区・戸塚区)横浜市(青葉区・旭区・瀬谷区・緑区)川崎市(麻生区・川崎区・幸区・高津区・多摩区・中原区・宮前区)鎌倉市・逗子市・横須賀市・三浦市・三浦郡(葉山町)綾瀬市・厚木市・伊勢原市・海老名市・相模原市・座間市・秦野市・大和市・愛甲郡(愛川町・清川村)・高座郡(寒川町)茅ヶ崎市・平塚市・藤沢市・中郡(大磯町)小田原市・南足柄市・足柄上郡(大井町・開成町・中井町・松田町・山北町)・足柄下郡(箱根町・真鶴町・湯河原町)・中郡(二宮町)


### 9.5.2 検索結果のプロンプトへの組み込み

#### OpenAI APIキーセットと動作確認

In [14]:
import os
from google.colab import userdata
openai_api_key = userdata.get("OPENAI_API_KEY")

In [15]:
import openai

openai.api_key = openai_api_key

completion = openai.ChatCompletion.create(
  model="gpt-4o-mini",
  messages=[
    {"role": "user", "content": "write a haiku about ai (japanese)"}
  ]
)

print(completion.choices[0].message)
response = completion.choices[0].message.content
print(response)

{
  "annotations": [],
  "content": "\u4eba\u5de5\u77e5\u80fd  \n\u672a\u6765\u3092\u7167\u3089\u3059  \n\u5149\u306e\u661f\u3088  ",
  "refusal": null,
  "role": "assistant"
}
人工知能  
未来を照らす  
光の星よ  


In [16]:
from abc import ABCMeta, abstractmethod

class PromptMaker(metaclass=ABCMeta):
    """クイズ用プロンプトを作成するための抽象クラス"""

    @abstractmethod
    def run(self, questions: list[str]) -> list[str]:
        """プロンプトの作成（具体的な実装は継承先で行われる）"""
        pass

class RetrievalPromptMaker(PromptMaker):
    """
    質問から関連するテキストを取得し、
    それを組み入れたプロンプトを作成する
    """
    def __init__(self, binary_k: int=2048, top_k: int=3):
        """初期化処理の定義"""
        self.binary_k = binary_k
        self.top_k = top_k

    def run(self, questions: list[str]) -> list[str]:
        """プロンプトを作成"""
        retrieved_passages_list = retrieve_passages(
            questions, binary_k=self.binary_k, top_k=self.top_k
        )
        retrieved_texts = [
            "\n".join(ps) for ps in retrieved_passages_list
        ]
        return [
            "あなたには今からクイズに答えてもらいます。"
            "問題を与えますので、その解答のみを簡潔に出力してください。\n"
            "また解答の参考になりうるテキストを与えます。"
            "解答を含まない場合もあるのでその場合は無視してください。\n"
            "\n"
            f"{text}\n"
            "\n"
            f"問題：{question}\n"
            "解答："
            for question, text in zip(questions, retrieved_texts)
        ]

In [17]:
retrieval_prompt_maker = RetrievalPromptMaker(top_k=3)
print(retrieval_prompt_maker.run(["日本で一番高い山は何？"])[0])

あなたには今からクイズに答えてもらいます。問題を与えますので、その解答のみを簡潔に出力してください。
また解答の参考になりうるテキストを与えます。解答を含まない場合もあるのでその場合は無視してください。

タイトル: 日本の山
本文: 日本の領土に占める山間部の割合はおよそ7割程度である。2020年現在、日本で一番高い山は富士山で、逆に一番低い山は日和山である。日本は新期造山帯に位置し、多くの火山が見られる。日本では昔から人々が山と共生する文化が培われてきた。それらの山は里山とよばれ、農村などでは山を共有地として村全体で管理し、薪をとったり土や山菜などを利用する目的で利用され、手入れをされてきた。しかし、明治時代になるとそれらの山の中には国有地にされてしまったものも多く存在する。また、日本の山の中には霊峰とよばれ、民間信仰の場にされた山も多くある。山の中には昔から金山や銀山として、近代では銅や石炭などを入手するため多くの鉱山が作られ、鉱毒や粉塵などが問題になった。特に近代において、山林の多くで木材を得るための大規模な伐採などが行われた。
タイトル: 日本の山一覧 (3000m峰)
本文: 3000 m峰は、独立峰の富士山と御嶽山及び飛騨山脈(北アルプス)と赤石山脈(南アルプス)の山域に限られている。24番目に高い山は剱岳 (2999 m) で、3000 mにわずか1 m届いていない。標高3000 mを越える一帯は森林限界のハイマツ帯で、高山植物の群生地となっている。また多くの3000 m峰のハイマツ帯は、ライチョウ(雷鳥)の生息地となっている。
タイトル: 一等三角点百名山
本文: 三角点は一般的に眺望の利く場所に設置され一等三角点は970点余りに上るが、その中で風格のある山容、優れた眺望、高い知名度、さらに概ね標高1,000m以上で登りがいのある山が選定された。一等三角点の最高峰は、標高3,121 mの南アルプスの赤石岳である。三角点がその山の最高峰とは限らず、山頂に三角点より高い場所があり、その地点すなわち標高点や測定点を以て山の高さとしている例も少なくない(例:早池峰、御嶽山、金剛山)。また連山を形成している場合、最高峰でないピークに一等三角点が設置されている例もある(例:赤城山、穂高岳、阿蘇山)。特殊な例としては雲仙普賢岳のように、噴火により一等三角点が最高

In [18]:
import asyncio
from typing import Awaitable, Callable, TypeVar
from openai.error import APIError # OpenAIErrorをAPIErrorに修正

T = TypeVar("T")

async def retry_on_error(
    openai_call: Callable[[], Awaitable[T]],
    max_num_trials: int = 5,
    first_wait_time: int = 10,
) -> Awaitable[Awaitable]:
    """OpenAI API使用時にエラーが帰ってきた場合に再試行する"""
    for i in range(max_num_trials):
        try:
            # 関数を実行する
            return await openai_call()
        except APIError as e: # OpenAIErrorをAPIErrorに修正
            # 試行回数が上限に達したらエラーを送出
            if i == max_num_trials - 1:
                raise
            print(f"エラーを受け取りました: {e}")
            wait_time_seconds = first_wait_time * (2**i)
            print(f"{wait_time_seconds}秒待機します")
            await asyncio.sleep(wait_time_seconds)

In [26]:
from torch.nn.modules import MSELoss
import openai

async def _async_batch_run_chatgpt(
    messages_list: list[list[dict[str, str]]],
    temperature: float,
    max_tokens: int | None,
    stop: str | list[str] | None,
) -> list[str]:
    """OpenAI APIに並列してリクエストを送る"""
    # コルーチンオブジェクトをtasksに格納
    tasks = [
        retry_on_error(
            # ラムダ式で無名関数を定義して渡し、retry_on_error関数の内部で呼び出させる
            openai_call=lambda x=ms: openai.ChatCompletion.acreate(
                model="gpt-4o-mini",
                messages=x,
                temperature=temperature,
                max_tokens=max_tokens,
                stop=stop,
            )
        )
            for ms in messages_list
    ]
    # tasks内の非同期処理を実行し結果を収集
    completions = await asyncio.gather(*tasks)
    return [
        c["choices"][0]["message"]["content"]
        for c in completions
    ]

def batch_run_chatgpt(
        messages_list: list[list[dict[str, str]]],
        temperature: float = 0.0,
        max_tokens: int | None = None,
        stop: str | list[str] | None = None,
) -> list[str]:
    """非同期処理関数を実行するためのラッパー"""
    return asyncio.run(
        _async_batch_run_chatgpt(
            messages_list, temperature, max_tokens, stop
        )
    )


In [27]:
import nest_asyncio

nest_asyncio.apply()

In [31]:
from datasets import Dataset
from tqdm import tqdm

def get_chatgpt_outputs_for_quiz(
    quiz_prompt_maker: PromptMaker,
    quiz_dataset: Dataset,
    batch_size: int,
) -> list[str]:
    """
    クイズ用のプロンプトを使用した際の
    データセットの核問題に対するモデルの回答を集める
    """
    output_answers: list[str] = []
    with tqdm(total=len(quiz_dataset)) as pbar:
        for batch in quiz_dataset.iter(batch_size=batch_size):
            # 入力の準備
            prompts = quiz_prompt_maker.run(batch["question"])
            inputs = [
                [{"role": "user", "content": p}] for p in prompts
            ]

            # APIにリクエストを送信
            answers = batch_run_chatgpt(inputs)

            # モデルの回答を表示
            for question, answer in zip(batch["question"], answers):
                print(f"問題: {question}")
                print(f"解答: {answer}")
                print()

            output_answers += answers
            pbar.update(len(answers))
    return output_answers

In [32]:
def calculate_quiz_accuracy(
    output_answers: list[str], correct_answers_list: list[list[str]]
) -> float:
    """モデルの解答と正解の解答例から正解率を算出する"""
    num_correct = 0
    for output_answer, answers in zip(
        output_answers, correct_answers_list
    ):
        # モデルの出力が解答例を一つでも含んでいれば正解とみなす
        num_correct += int(any(a in output_answer for a in answers))
    return num_correct / len(output_answers)

In [23]:
quiz_dataset = load_dataset("llm-book/aio", split="validation")

README.md: 0.00B [00:00, ?B/s]

aio.py: 0.00B [00:00, ?B/s]

default/train/0000.parquet:   0%|          | 0.00/5.10M [00:00<?, ?B/s]

default/validation/0000.parquet:   0%|          | 0.00/238k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22335 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [33]:
retrieval_output_answers = get_chatgpt_outputs_for_quiz(
    retrieval_prompt_maker, quiz_dataset, batch_size=4
)
retrieval_accuracy = calculate_quiz_accuracy(
    retrieval_output_answers,
    [item["answers"] for item in quiz_dataset],
)
print(f"正解率: {retrieval_accuracy * 100}")

  0%|          | 4/1000 [00:01<07:41,  2.16it/s]

問題: 映画『ウエスト・サイド物語』に登場する2つの少年グループといえば、シャーク団と何団?
解答: ジェット団

問題: 氷った海に穴を開けて漁をすることから、漢字で「氷の下の魚」と書くタラ科の魚は何?
解答: コマイ

問題: アメリカとロシア間で結ばれた「戦略兵器削減条約」のことをアルファベット5文字の略称で何という?
解答: START

問題: リンゴが木から落ちる様子を見て「万有引力」を発見したといわれる、17世紀のイギリスの科学者は誰?
解答: アイザック・ニュートン



  1%|          | 8/1000 [00:02<05:07,  3.23it/s]

問題: 聖武天皇の時代に栄えた、東大寺正倉院や唐招提寺金堂など、中国・唐の影響を強く受けた貴族文化を何という?
解答: 天平文化

問題: アメリカ・メジャーリーグの2つのリーグといえば、ナショナルリーグと何リーグ?
解答: アメリカンリーグ

問題: 池坊、草月流、小原流といえば、どんな分野の三大流派?
解答: いけばな

問題: 英語で「取り巻く」という意味がある、前後左右から音が聞こえる立体音響を何という?
解答: バイノーラル音響



  1%|          | 12/1000 [00:03<05:03,  3.25it/s]

...

問題: 「冬の大三角」を形成する3つの星は、おおいぬ座のシリウス、こいぬ座のプロキオンとオリオン座の何?
解答: ベテルギウス

問題: 江戸時代末期・大坂に洗心堂という私塾を開いた儒学者で、1837年に江戸幕府に対して反乱を起こすも鎮圧され自害した人物は誰?
解答: 大塩平八郎

問題: 『ブエノスアイレス午前零時』で芥川賞を受賞したのは藤沢周ですが、『暗殺の年輪』で直木賞を受賞したのは誰?
解答: 藤沢周平

問題: 手紙やハガキを送った際に、料金を支払った証拠として郵便局で押される、日付入りのスタンプのことを何という?
解答: 消印



100%|█████████▉| 996/1000 [06:01<00:01,  3.52it/s]

問題: NBAや格闘技などでみられる、選手が対戦相手に対して挑発的な言葉を発して心理的な揺さぶりをかける戦術を何という?
解答: トラッシュ・トーク

問題: いい加減で役立たずな人のことを、落語によく登場する間抜けな人物にたとえて何という?
解答: 愚か者

問題: パーソン、パンサー、ジャクソンなどの種類がいる、周囲の環境に合わせて体の色を変えることでおなじみの動物といえば何?
解答: カメレオン

問題: 小説『古都』や『伊豆の踊子』などの作品を書き、1968年にノーベル文学賞を受賞した作家は誰?
解答: 川端康成



100%|██████████| 1000/1000 [06:02<00:00,  2.76it/s]

問題: いわゆる「古典の三大随筆」と呼ばれる3つの作品といえば、『枕草子』、『方丈記』と何?
解答: 『徒然草』

問題: 正式名称を「新幹線電気軌道総合試験車」という、新幹線の線路や架線の状態を調査するための車両を車体の色から何という?
解答: ドクターイエロー

問題: 明治11年、東京・紀尾井坂で石川県士族の島田一郎らに暗殺された政治家は誰?
解答: 大久保利通

問題: 中国を代表するIT企業の総称「BATH」で表される4つの会社といえば、Baidu、Alibaba、Tencentと何?
解答: Huawei

正解率: 75.2
